# MTMC 11b \u2014 WILDTRACK Stage 3 (FAISS Indexing)

Builds FAISS index over person ReID embeddings from 11a.

**Prerequisite**: attach **11a's output** as a data source:
`Add Data -> Kernel Output -> search "mtmc-11a-wildtrack-stages-0-2" -> add`

| Stage | What | Time |
|---|---|---|
| 3 | Build FAISS similarity index over ReID features | ~1 min |

After this runs, attach **this** notebook's output to **11c**.

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile
from pathlib import Path

# --- Guard: detect GPU BEFORE importing torch ---
# Kaggle's PyTorch 2.10+ drops P100 (sm_60) support.
# If we got a P100, downgrade to a compatible build first.
if shutil.which("nvidia-smi"):
    _nvsmi = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True)
    if _nvsmi.returncode == 0 and _nvsmi.stdout.strip():
        _gpu_name, _cap = _nvsmi.stdout.strip().split(",", 1)
        _major, _minor = _cap.strip().split(".")
        _sm = int(_major) * 10 + int(_minor)
        if _sm < 70:
            print(f"\u26a0 GPU {_gpu_name.strip()} (sm_{_sm}) \u2014 installing compatible PyTorch ...")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "torch==2.4.1", "torchvision==0.19.1",
                "--index-url", "https://download.pytorch.org/whl/cu124",
            ])
            print("\u2713 Compatible PyTorch installed")

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## 1. Clone Repo & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/abdoibrahim257/MTC-repo.git"
PROJECT = Path("/kaggle/working/gp")

if not PROJECT.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)])
os.chdir(str(PROJECT))

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "faiss-cpu>=1.7", "omegaconf>=2.3", "networkx>=3.1",
    "scipy", "scikit-learn", "click", "tqdm"])
print("\u2713 Dependencies installed")

## 2. Load Checkpoint from 11a

In [ ]:
DATA_OUT = Path("/tmp/pipeline_outputs")
DATA_OUT.mkdir(parents=True, exist_ok=True)

# Find 11a output
INPUT_11A = None
for candidate in [
    Path("/kaggle/input/mtmc-11a-wildtrack-stages-0-2"),
    Path("/kaggle/input/mtmc-11a-wildtrack-stages-0-2-tracking-reid"),
]:
    if candidate.exists():
        INPUT_11A = candidate
        break

if INPUT_11A is None:
    for d in Path("/kaggle/input").iterdir():
        if (d / "checkpoint.tar.gz").exists():
            INPUT_11A = d
            break

if INPUT_11A is None:
    raise FileNotFoundError("11a output not found in /kaggle/input/")

ckpt = INPUT_11A / "checkpoint.tar.gz"
assert ckpt.exists(), f"checkpoint.tar.gz not found at {ckpt}"
print(f"Checkpoint: {ckpt} ({ckpt.stat().st_size/1024**2:.1f} MB)")

with tarfile.open(str(ckpt), "r:gz") as tar:
    tar.extractall(path=str(DATA_OUT))
    print(f"Extracted to {DATA_OUT}")

# Read metadata
meta_path = DATA_OUT / "run_metadata.json"
with open(meta_path) as f:
    meta = json.load(f)
RUN_NAME = meta["run_name"]
print(f"\u2713 Run name: {RUN_NAME}")

run_dir = DATA_OUT / RUN_NAME
for stage in ["stage0", "stage1", "stage2"]:
    sd = run_dir / stage
    if sd.exists():
        nf = sum(1 for _ in sd.rglob("*") if _.is_file())
        print(f"  {stage}: {nf} files")
    else:
        print(f"  {stage}: not present")

## 3. Run Stage 3 (FAISS Indexing)

In [ ]:
os.chdir(str(PROJECT))

cmd = [
    sys.executable, "scripts/run_pipeline.py",
    "--config", "configs/default.yaml",
    "--dataset-config", "configs/datasets/wildtrack.yaml",
    "--stages", "3",
    "--override", f"project.run_name={RUN_NAME}",
    "--override", f"project.output_dir={DATA_OUT}",
]

print(f"CMD: {' '.join(str(c) for c in cmd)}")
print("=" * 70)
t0 = time.time()
r = subprocess.run(cmd, cwd=str(PROJECT))
elapsed = time.time() - t0
if r.returncode != 0:
    print(f"\u2717 FAILED after {elapsed:.1f}s")
    sys.exit(r.returncode)
print("=" * 70)
print(f"\u2713 Stage 3 done in {elapsed:.1f}s")

stage3_dir = DATA_OUT / RUN_NAME / "stage3"
if stage3_dir.exists():
    nf = sum(1 for _ in stage3_dir.rglob("*") if _.is_file())
    sz = sum(f.stat().st_size for f in stage3_dir.rglob("*") if f.is_file()) / 1024**2
    print(f"  stage3: {nf} files ({sz:.1f} MB)")

## 4. Save Checkpoint for 11c

In [ ]:
checkpoint_path_out = Path("/kaggle/working/checkpoint.tar.gz")
metadata_path_out = Path("/kaggle/working/run_metadata.json")
with open(metadata_path_out, "w") as f:
    json.dump({"run_name": RUN_NAME}, f)

with tarfile.open(str(checkpoint_path_out), "w:gz") as tar:
    tar.add(str(metadata_path_out), arcname="run_metadata.json")

    for stage in ["stage1", "stage2", "stage3"]:
        stage_dir = DATA_OUT / RUN_NAME / stage
        if stage_dir.exists():
            n = 0
            for fpath in stage_dir.rglob("*"):
                if fpath.is_file():
                    tar.add(str(fpath), arcname=f"{RUN_NAME}/{stage}/{fpath.relative_to(stage_dir)}")
                    n += 1
            print(f"  + {stage}/ ({n} files)")

    # Forward GT annotations
    gt_dir = DATA_OUT / "gt_annotations"
    if gt_dir.exists():
        n = 0
        for fpath in gt_dir.rglob("*"):
            if fpath.is_file():
                tar.add(str(fpath), arcname=f"gt_annotations/{fpath.relative_to(gt_dir)}")
                n += 1
        print(f"  + gt_annotations/ ({n} files forwarded)")

sz = checkpoint_path_out.stat().st_size / 1024**2
print(f"\n\u2713 Checkpoint: {checkpoint_path_out}  ({sz:.1f} MB)")
print("  Next: attach this notebook's output to 11c, then push 11c.")